# Day 3: Build a Web App for Student Score Prediction
## From Python Model to Working Website

**What We Will Learn Today:**
1. What is Flask? (Simple web framework)
2. Load our saved model
3. Create a web page with a form
4. Show predictions on the website
5. Make it look nice with simple HTML

**Our Goal:** Create a website where anyone can enter study hours and see predicted scores!

## 🌐 What is Flask?

**Simple Explanation:**

Flask is a tool that lets us turn Python code into a website.

```
User opens website → Enters study hours → Clicks button → Flask sends to model → Shows prediction
```

**Think of it like:**
- Flask is the WAITER in a restaurant
- You (user) give your order (study hours)
- Waiter takes to kitchen (our model)
- Kitchen makes food (makes prediction)
- Waiter brings back to you (shows prediction)

**This is called a Web Application!**

## 🔧 Step 1: Install Flask

First, we need to install Flask (only need to do this once)

In [1]:
# Install Flask (uncomment and run once)
# !pip install flask

print("📦 Checking Flask installation...")
try:
    import flask
    print(f"✅ Flask is installed! Version: {flask.__version__}")
except:
    print("❌ Flask not found. Run: !pip install flask")
    print("\nIf you're in Jupyter Notebook, you need to create a .py file for Flask.")
    print("We'll do that in the next step!")

📦 Checking Flask installation...
✅ Flask is installed! Version: 3.1.3


/var/folders/2f/kh36lv3s5j16g02h8g82vnkc0000gn/T/ipykernel_16758/834799055.py:7: DeprecationWarning: The '__version__' attribute is deprecated and will be removed in Flask 3.2. Use feature detection or 'importlib.metadata.version("flask")' instead.
  print(f"✅ Flask is installed! Version: {flask.__version__}")


## 📁 Step 2: Create Project Files

Flask apps need specific files. Let's create them:

In [2]:
# Create the templates folder for HTML files
if not os.path.exists('student_predictor_app/templates'):
    os.makedirs('student_predictor_app/templates')
    print("✅ Created templates folder")

print("\n📁 Updated structure:")
print("   student_predictor_app/")
print("   ├── simple_student_model.pkl")
print("   └── templates/")

NameError: name 'os' is not defined

## 🌐 Step 3: Create the Web Page (HTML)

In [3]:
# This is the HTML code for our website
html_content = """
<!DOCTYPE html>
<html>
<head>
    <title>Student Score Predictor</title>
    <style>
        body {
            font-family: Arial, sans-serif;
            max-width: 600px;
            margin: 50px auto;
            padding: 20px;
            background-color: #f0f7ff;
        }
        .container {
            background-color: white;
            padding: 30px;
            border-radius: 10px;
            box-shadow: 0 2px 10px rgba(0,0,0,0.1);
        }
        h1 {
            color: #2c3e50;
            text-align: center;
        }
        label {
            font-weight: bold;
            display: block;
            margin-top: 20px;
        }
        input {
            width: 100%;
            padding: 10px;
            margin: 10px 0;
            border: 2px solid #3498db;
            border-radius: 5px;
            font-size: 16px;
        }
        button {
            background-color: #3498db;
            color: white;
            padding: 12px 24px;
            border: none;
            border-radius: 5px;
            cursor: pointer;
            font-size: 16px;
            width: 100%;
        }
        button:hover {
            background-color: #2980b9;
        }
        .result {
            margin-top: 30px;
            padding: 20px;
            background-color: #e8f4fd;
            border-radius: 5px;
            text-align: center;
        }
        .score {
            font-size: 48px;
            font-weight: bold;
            color: #2c3e50;
        }
        .grade {
            font-size: 24px;
            margin-top: 10px;
        }
        .tip {
            margin-top: 20px;
            padding: 15px;
            background-color: #fff3cd;
            border-radius: 5px;
        }
        .footer {
            text-align: center;
            margin-top: 30px;
            color: #7f8c8d;
            font-size: 12px;
        }
    </style>
</head>
<body>
    <div class="container">
        <h1>🎓 Student Score Predictor</h1>
        <p style="text-align: center;">Enter how many hours you study each day to predict your final score!</p>
        
        <form action="/predict" method="POST">
            <label for="hours">📚 Hours Studied per Day:</label>
            <input type="number" step="0.5" name="hours" id="hours" 
                   placeholder="Enter 1-12 hours" min="0" max="15" required>
            <button type="submit">🔮 Predict My Score</button>
        </form>
        
        {result_html}
        
        <div class="footer">
            <p>💡 This prediction is based on historical student data</p>
            <p>📊 Model accuracy: ±{accuracy} points</p>
        </div>
    </div>
</body>
</html>
"""

# Save the HTML template
with open('student_predictor_app/templates/index.html', 'w') as f:
    f.write(html_content)

print("✅ Created web page template: index.html")
print("   Location: student_predictor_app/templates/index.html")

FileNotFoundError: [Errno 2] No such file or directory: 'student_predictor_app/templates/index.html'

## 🐍 Step 4: Create the Flask App (The Brain)

In [ ]:
# This is the Python code for our Flask application
flask_app_content = """
from flask import Flask, render_template, request
import pickle
import numpy as np

# Create Flask app
app = Flask(__name__)

# Load our trained model
with open('simple_student_model.pkl', 'rb') as f:
    model = pickle.load(f)

# Get model formula (for display)
coefficient = model.coef_[0]
intercept = model.intercept_

# Function to determine grade
def get_grade(score):
    if score >= 90:
        return ("A", "🌟 Excellent! Keep up the great work!", "#27ae60")
    elif score >= 80:
        return ("B", "👍 Good job! You're doing well!", "#3498db")
    elif score >= 70:
        return ("C", "📚 Good! With a little more effort, you can do even better!", "#f39c12")
    elif score >= 60:
        return ("D", "⚠️ You're passing, but try to study more!", "#e67e22")
    else:
        return ("F", "💪 Don't give up! Try increasing your study hours!", "#e74c3c")

# Function to give study tips
def get_tip(hours, score):
    if score < 60:
        return f"💡 Tip: Try studying {max(5, hours + 2):.1f} hours per day to improve your score!"
    elif score < 75:
        return "💡 Tip: You're on the right track! Stay consistent with your studies."
    else:
        return "🎯 Tip: Great work! Consider helping other students who are struggling."

# Home page
@app.route('/')
def home():
    # Show empty form
    return render_template('index.html', 
                          result_html='',
                          accuracy=round(5, 1))

# Prediction page
@app.route('/predict', methods=['POST'])
def predict():
    # Get hours from form
    hours = float(request.form['hours'])
    
    # Make prediction
    predicted_score = model.predict([[hours]])[0]
    
    # Round to 1 decimal
    predicted_score = round(predicted_score, 1)
    
    # Get grade and advice
    grade, message, color = get_grade(predicted_score)
    tip = get_tip(hours, predicted_score)
    
    # Create HTML for results
    result_html = f"""
    <div class="result">
        <h2>Your Predicted Score:</h2>
        <div class="score" style="color: {color};">{predicted_score}</div>
        <div class="grade">Grade: {grade}</div>
        <p>{message}</p>
        <div class="tip">
            {tip}
        </div>
    </div>
    """
    
    return render_template('index.html', 
                          result_html=result_html,
                          accuracy=round(5, 1))

# Run the app
if __name__ == '__main__':
    print("🚀 Starting Student Score Predictor...")
    print(f"📊 Model: Score = {coefficient:.2f} × hours + {intercept:.2f}")
    print("🌐 Open your browser and go to: http://127.0.0.1:5000")
    app.run(debug=True)
"""

# Save the Flask app
with open('student_predictor_app/app.py', 'w') as f:
    f.write(flask_app_content)

print("✅ Created Flask app: app.py")
print("   Location: student_predictor_app/app.py")
print("\n📁 Final project structure:")
print("   student_predictor_app/")
print("   ├── app.py")
print("   ├── simple_student_model.pkl")
print("   └── templates/")
print("       └── index.html")

## 🚀 Step 5: Run Your Web App

**Now for the exciting part - let's run our website!**

In [ ]:
# Instructions to run the app
print("="*60)
print("🚀 HOW TO RUN YOUR WEB APP")
print("="*60)

print("""
Method 1: Using Command Line / Terminal
========================================
1. Open Terminal (Mac/Linux) or Command Prompt (Windows)
2. Navigate to your app folder:
   cd student_predictor_app
3. Run the app:
   python app.py
4. Open your browser and go to: http://127.0.0.1:5000

Method 2: Using the code below (click Run)
============================================
""")

# Option to run directly from notebook (may require restart)
print("\n⚠️ Note: Flask apps run continuously.")
print("   To stop the app, press Ctrl+C in the terminal.")
print("\n📱 Once running, you can:")
print("   - Share the link with friends")
print("   - Let others on your network access it")
print("   - Keep it running as a web service")

print("\n" + "="*60)
print("✅ All files are ready!")
print("="*60)
print("\n📁 Your files are in: student_predictor_app/")
print("   - app.py (the Flask application)")
print("   - templates/index.html (the webpage)")
print("   - simple_student_model.pkl (your trained model)")

## 🎨 Step 6: Understanding How It Works

In [ ]:
print("="*60)
print("🔍 HOW THE WEB APP WORKS")
print("="*60)

explanation = {
    "1. User visits website": "http://127.0.0.1:5000",
    "2. Flask sends HTML page": "User sees the form",
    "3. User enters study hours": "Example: 6 hours",
    "4. Clicks 'Predict' button": "Form data sent to Flask",
    "5. Flask loads the model": "Uses pickle to load .pkl file",
    "6. Model makes prediction": "Score = coefficient × 6 + intercept",
    "7. Flask creates result page": "Shows prediction with grade",
    "8. User sees result": "'Your score will be 78.5'"
}

for step, description in explanation.items():
    print(f"{step}:")
    print(f"   → {description}")
    print()

print("💡 Think of it as:")
print("   User → Website → Flask → Model → Prediction → Back to User")
print("   (Order)  (View)    (Waiter)  (Chef)   (Food)     (Serve)")

## 🎯 Step 7: Try Your App (Interactive Example)

Let's simulate what your web app does (without actually running the server)

In [ ]:
# Load the model
import pickle

with open('student_predictor_app/simple_student_model.pkl', 'rb') as f:
    demo_model = pickle.load(f)

coefficient = demo_model.coef_[0]
intercept = demo_model.intercept_

print("="*60)
print("🎓 STUDENT SCORE PREDICTOR (Demo)")
print("="*60)
print(f"\n📐 Model Formula: Score = {coefficient:.2f} × Hours + {intercept:.2f}")
print("\n" + "-"*60)

# Interactive demo
while True:
    try:
        hours = input("\n📚 Enter study hours per day (or 'quit' to exit): ")
        if hours.lower() == 'quit':
            print("\n👋 Thanks for using the predictor!")
            break
        
        hours = float(hours)
        if hours < 0 or hours > 15:
            print("⚠️ Please enter a number between 0 and 15")
            continue
        
        # Make prediction
        predicted = demo_model.predict([[hours]])[0]
        predicted = round(predicted, 1)
        
        # Determine grade
        if predicted >= 90:
            grade = "A"
            emoji = "🌟"
            advice = "Excellent! Keep up the great work!"
        elif predicted >= 80:
            grade = "B"
            emoji = "👍"
            advice = "Good job! You're doing well!"
        elif predicted >= 70:
            grade = "C"
            emoji = "📚"
            advice = "Good! With more effort, you can do even better!"
        elif predicted >= 60:
            grade = "D"
            emoji = "⚠️"
            advice = "You're passing, but try to study more!"
        else:
            grade = "F"
            emoji = "💪"
            advice = f"Try increasing your study hours to {max(5, hours + 2):.1f} hours/day!"
        
        # Show result
        print("\n" + "="*40)
        print(f"{emoji} YOUR PREDICTION:")
        print(f"   📊 Predicted Score: {predicted}")
        print(f"   🎓 Grade: {grade}")
        print(f"   💡 Tip: {advice}")
        print("="*40)
        
    except ValueError:
        print("❌ Please enter a valid number!")

print("\n" + "="*60)
print("🎉 That's exactly what your web app will do!")
print("="*60)

## 📱 Step 8: Making It Better - Add More Features

In [ ]:
# Let's create an improved version with multiple features

improved_app_content = """
from flask import Flask, render_template, request
import pickle
import numpy as np

app = Flask(__name__)

# Load model
with open('simple_student_model.pkl', 'rb') as f:
    model = pickle.load(f)

@app.route('/')
def home():
    return render_template('improved.html', prediction=None)

@app.route('/predict', methods=['POST'])
def predict():
    # Get all inputs
    hours = float(request.form['hours'])
    previous = float(request.form['previous'])
    attendance = float(request.form['attendance'])
    
    # Simple formula (in real app, use trained model)
    score = (hours * 2.5) + (previous * 0.4) + (attendance * 0.2)
    score = round(score, 1)
    
    # Grade logic
    if score >= 90:
        grade = "A"
        color = "green"
    elif score >= 80:
        grade = "B"
        color = "blue"
    elif score >= 70:
        grade = "C"
        color = "orange"
    elif score >= 60:
        grade = "D"
        color = "orange"
    else:
        grade = "F"
        color = "red"
    
    return render_template('improved.html', 
                          prediction=score, 
                          grade=grade,
                          color=color,
                          hours=hours,
                          previous=previous,
                          attendance=attendance)

if __name__ == '__main__':
    app.run(debug=True)
"""

# Create improved HTML
improved_html = """
<!DOCTYPE html>
<html>
<head>
    <title>Advanced Score Predictor</title>
    <style>
        body { font-family: Arial; max-width: 500px; margin: 50px auto; padding: 20px; }
        input { width: 100%; padding: 8px; margin: 10px 0; }
        button { background: #4CAF50; color: white; padding: 10px; width: 100%; }
        .result { margin-top: 20px; padding: 20px; background: #f0f0f0; text-align: center; }
        .score { font-size: 48px; font-weight: bold; }
    </style>
</head>
<body>
    <h1>🎓 Advanced Score Predictor</h1>
    <form method="POST">
        <label>📚 Hours Studied per Day:</label>
        <input type="number" step="0.5" name="hours" required>
        
        <label>📊 Previous Score:</label>
        <input type="number" step="1" name="previous" required>
        
        <label>📈 Attendance (%):</label>
        <input type="number" step="1" name="attendance" required>
        
        <button type="submit">Predict Score</button>
    </form>
    
    {% if prediction %}
    <div class="result">
        <h2>Your Predicted Score:</h2>
        <div class="score" style="color: {{color}};">{{ prediction }}</div>
        <div>Grade: {{ grade }}</div>
        <hr>
        <small>Based on: {{ hours }} hours study, {{ previous }} previous score, {{ attendance }}% attendance</small>
    </div>
    {% endif %}
</body>
</html>
"""

# Save improved version
with open('student_predictor_app/templates/improved.html', 'w') as f:
    f.write(improved_html)

print("✅ Created improved version with multiple features!")
print("   File: student_predictor_app/templates/improved.html")
print("\n💡 This version uses:")
print("   - Study hours")
print("   - Previous score")
print("   - Attendance percentage")

## 📝 SIMPLE EXERCISES

### Exercise 1: Modify the Message

**Task:** Change the tip message for students who score above 85 to say "You're a star student!"

In [ ]:
# EXERCISE 1 SOLUTION
def get_custom_tip(score):
    if score >= 85:
        return "🌟 You're a star student! Keep shining!"
    elif score >= 70:
        return "📚 You're doing great! Aim for 85+!"
    else:
        return "💪 Keep working hard! Every hour counts!"

# Test
print("Tips for different scores:")
for score in [95, 75, 55]:
    print(f"  Score {score}: {get_custom_tip(score)}")

### Exercise 2: Add a New Feature

**Task:** Add "sleep hours" as a new input field in the improved version

In [ ]:
# EXERCISE 2 SOLUTION
print("Add this to your improved.html form:")
print("""
<label>😴 Sleep Hours per Night:</label>
<input type="number" step="0.5" name="sleep" required>
""")

print("\nThen add to app.py:")
print("""
sleep = float(request.form['sleep'])
score = (hours * 2.5) + (previous * 0.4) + (attendance * 0.2) + (sleep * 1.2)
""")

print("\n✅ Now sleep quality affects the prediction!")

### Exercise 3: Add a Color Coded Result

**Task:** Make the result box change color based on score

In [ ]:
# EXERCISE 3 SOLUTION
def get_box_color(score):
    if score >= 90:
        return "#d4edda"  # Light green
    elif score >= 70:
        return "#fff3cd"  # Light yellow
    else:
        return "#f8d7da"  # Light red

print("Color suggestions:")
print(f"  Score 95 → {get_box_color(95)} (Green for excellent)")
print(f"  Score 75 → {get_box_color(75)} (Yellow for average)")
print(f"  Score 55 → {get_box_color(55)} (Red for needs improvement)")

print("\n💡 To implement: Add style='background-color: {{box_color}}' to result div")

### Exercise 4: Share Your App

**Task:** Make your app accessible to others on the same network

In [ ]:
# EXERCISE 4 SOLUTION
print("To share your app with others on the same Wi-Fi:")
print("\n1. Change this line in app.py:")
print("   FROM: app.run(debug=True)")
print("   TO:   app.run(host='0.0.0.0', debug=True)")

print("\n2. Find your computer's IP address:")
print("   Windows: ipconfig (look for IPv4 Address)")
print("   Mac/Linux: ifconfig or ip addr")

print("\n3. Others can access at: http://YOUR_IP:5000")
print("\n📱 Example: http://192.168.1.100:5000")

print("\n⚠️ Make sure your firewall allows port 5000!")

## 🚀 ASSIGNMENT: Complete Your Web App

**Your Task:** Build a complete student performance predictor website

**Requirements:**
1. Create a welcoming homepage
2. Accept at least 3 inputs (study hours, previous score, attendance)
3. Show prediction with grade
4. Give personalized advice
5. Make it look nice with colors
6. Add a "Reset" button
7. Show the prediction formula used

**Bonus:**
- Add a progress bar
- Add emojis based on score
- Save prediction history
- Add a dark mode toggle

In [ ]:
# ASSIGNMENT SOLUTION TEMPLATE
complete_app_template = """
from flask import Flask, render_template, request
import pickle

app = Flask(__name__)

# Load your trained model
# with open('simple_student_model.pkl', 'rb') as f:
#     model = pickle.load(f)

@app.route('/')
def home():
    return render_template('complete.html', result=None)

@app.route('/predict', methods=['POST'])
def predict():
    # Get form data
    hours = float(request.form['hours'])
    previous = float(request.form['previous'])
    attendance = float(request.form['attendance'])
    
    # Make prediction (use your trained model here)
    score = (hours * 2.5) + (previous * 0.4) + (attendance * 0.2)
    
    # Add noise for realism
    import random
    score += random.uniform(-3, 3)
    score = max(0, min(100, round(score, 1)))
    
    # Determine grade and advice
    if score >= 90:
        grade = "A"
        advice = "Outstanding! You're a top performer!"
        emoji = "🏆"
    elif score >= 80:
        grade = "B"
        advice = "Great job! You're above average!"
        emoji = "🎉"
    elif score >= 70:
        grade = "C"
        advice = "Good work! Aim for 80+ next time!"
        emoji = "📚"
    elif score >= 60:
        grade = "D"
        advice = "You passed! Try to study 1-2 hours more daily."
        emoji = "⚠️"
    else:
        grade = "F"
        advice = f"Don't give up! Try studying {max(5, hours+2)} hours per day."
        emoji = "💪"
    
    return render_template('complete.html', 
                          result=score,
                          grade=grade,
                          advice=advice,
                          emoji=emoji,
                          hours=hours,
                          previous=previous,
                          attendance=attendance)

if __name__ == '__main__':
    app.run(debug=True)
"""

print("✅ Assignment template created!")
print("\n📝 Your task is to:")
print("   1. Create the complete.html template")
print("   2. Add CSS styling")
print("   3. Test with different inputs")
print("   4. Share with classmates!")

## 🎓 Day 3 Summary

In [ ]:
print("="*60)
print("🎉 DAY 3 COMPLETE!")
print("="*60)

print("\n✅ What We Built Today:")
print("   1. Flask web application")
print("   2. HTML/CSS user interface")
print("   3. Model integration")
print("   4. Interactive predictions")
print("   5. Personalized feedback")

print("\n📁 Files Created:")
print("   📂 student_predictor_app/")
print("   ├── 📄 app.py (Flask application)")
print("   ├── 📄 simple_student_model.pkl (Trained model)")
print("   └── 📂 templates/")
print("       ├── 📄 index.html (Main webpage)")
print("       └── 📄 improved.html (Advanced version)")

print("\n🚀 To Run Your App:")
print("   1. Open terminal in student_predictor_app folder")
print("   2. Run: python app.py")
print("   3. Open browser: http://127.0.0.1:5000")

print("\n🎯 Next Steps:")
print("   • Add more features to your app")
print("   • Deploy online using PythonAnywhere or Render")
print("   • Share with friends and family")
print("   • Collect feedback and improve")

print("\n" + "="*60)
print("🌟 CONGRATULATIONS! You built a complete ML web app!")
print("="*60)
print("\nFrom Python basics → ML model → Working website!")
print("You're now a Full-Stack ML Developer! 🎉")